# fast_exp_analytics - example

This notebook shows a basic library scenario:

- AB analysis of the experiment
- ABC analysis of the experiment
- Duration planning: MDE, required duration, target MDE
- Excel export of results
- Preparing a chat message
- Optional LLM-review

In [2]:
import sys

sys.path.append("../../fast_exp_analytics")

In [3]:
import os
import pandas as pd

from fast_exp_analytics import (
    # ABC
    run_abc_test,
    style_table_abc,
    make_synthetic_abc_dataset,
    export_abc_results_to_excel,
    build_abc_chat_message,
    build_dashboard_url_abc,
    # AB
    run_ab_test,
    style_table_ab,
    export_ab_results_to_excel,
    build_ab_chat_message,
    build_dashboard_url_ab,
    # Messaging / LLM
    ChatSendConfig,
    send_chat_message,
    OpenAICompatConfig,
    build_llm_review,
    # Duration planning
    build_experiment_level,
    units_per_day_from_raw,
    default_duration_metrics_config,
    mde_table_from_aggregated_df,
    required_days_from_aggregated_df,
    mde_table_for_experiment_duration,
    required_days_for_target_mde_table,
    duration_plan_summary,
    export_duration_results_to_excel,
)

## Выгружаем исходные данные

(Пишем SQL/YQL/CHYT скрипты, чтобы получить данные для расчета)

In [4]:
# Генерация синтетического датасета для примера
df = make_synthetic_abc_dataset(n=100_000, seed=42)

In [5]:
df_ab = df[df["exp_group"] != "C"].copy()  # Синтетический датасет для AB
df_abc = df.copy()  # Синтетический датасет для ABC

In [6]:
df_ab.sample(5, random_state=42)

,exp_group,user_id,is_create_ad,is_with_payments,is_with_spents,campaigns,campaigns_with_spents,amount,a_amount_payment,shows,clicks,goals,camp_days_total,camp_days_avg,camp_days_max,camp_life_days,amount_per_day,goals_per_day,is_in_exp,amount_1000
77300,A,653539,1,0,1,3,1,0.0000,NaN,0,0,0,NaN,NaN,NaN,0.0000,0.0000,0.0000,1,0.0000
73309,B,892854,1,0,1,6,1,0.0000,NaN,116,0,0,NaN,4.4131,1.2869,0.0000,0.0000,0.0000,1,0.0000
91824,A,852007,1,0,1,1,4,74.8916,NaN,0,0,0,2.0929,NaN,NaN,2.0929,35.7836,0.0000,1,"74,891.6000"
96859,B,1246533,1,0,1,5,4,0.0000,NaN,0,0,0,NaN,NaN,NaN,0.0000,0.0000,0.0000,1,0.0000
70515,A,735246,1,0,1,1,1,44.3303,17.7607,0,0,0,NaN,NaN,NaN,0.0000,44.3303,0.0000,1,"44,330.3000"


In [7]:
df_abc.sample(5, random_state=228)

,exp_group,user_id,is_create_ad,is_with_payments,is_with_spents,campaigns,campaigns_with_spents,amount,a_amount_payment,shows,clicks,goals,camp_days_total,camp_days_avg,camp_days_max,camp_life_days,amount_per_day,goals_per_day,is_in_exp,amount_1000
80393,C,1221925,0,1,1,0,1,0.0000,NaN,0,0,0,NaN,NaN,NaN,0.0000,0.0000,0.0000,1,0.0000
90479,C,1412895,1,0,1,6,2,0.0000,74.9888,0,0,0,NaN,NaN,NaN,0.0000,0.0000,0.0000,1,0.0000
96406,A,680422,1,0,0,5,0,0.0000,NaN,0,0,0,5.8692,6.7164,11.5217,5.8692,0.0000,0.0000,1,0.0000
6909,B,682417,1,0,1,1,1,0.0000,NaN,0,0,26,NaN,NaN,NaN,0.0000,0.0000,26.0000,1,0.0000
48361,B,587577,1,0,1,2,3,0.0000,NaN,0,0,0,NaN,NaN,NaN,0.0000,0.0000,0.0000,1,0.0000


## 0. Общие настройки

In [ ]:
chat_id = "example-chat-id"
BOT = os.getenv("CHAT_BOT_TOKEN", "")
experiment_id = 4242
experiment_desc = "Тестируем пользователей удобного мессенджера"
exp_start_date = '2026-03-17'
exp_end_date = "2026-03-24"
base_url_ab = "https://example.com/dashboards/ab"
base_url_abс ="https://example.com/dashboards/abc"

# Конфиг чата передаётся снаружи библиотеки
chat_config = ChatSendConfig(
    api_base_url="https://api/bot/v1",
    token=BOT,
    timeout=30,
)


openai_api_key =  os.getenv("OPENAI_API_KEY", "")
openai_api_base = "https://openai/v1"
llm_config = OpenAICompatConfig(
    api_key=openai_api_key,
    base_url=openai_api_base,
    model="llm",
    timeout=60,
)

SYSTEM_PROMPT = os.getenv("SYSTEM_PROMT_LAYER", "")

## 1. AB example

### 1.1 Расчет

In [9]:
# Передаем метрики, которые будем считать. Делаем настройки. Задаем тип метрики. Все как ниже
# Здесь можно менять:
# - отображаемое название метрики
# - тип расчёта
# - numerator / denominator
# - направление улучшения

METRICS_DF = pd.DataFrame.from_dict(
    {
        "shows": ["Показы", "additive", "shows", "shows", "positive"],
        "clicks": ["Клики", "additive", "clicks", "clicks", "positive"],
        "amount": ["Списания", "additive", "amount", "amount", "positive"],
        "a_amount_payment": [
            "Живые пополнения",
            "average",
            "a_amount_payment",
            "a_amount_payment",
            "positive",
        ],
        "cpm": ["CPM", "ratio", "amount_1000", "shows", "negative"],
        "ctr": ["CTR", "ratio", "clicks", "shows", "positive"],
        "cpc": ["CPC", "ratio", "amount", "clicks", "negative"],
        "goals": ["Цели", "additive", "goals", "goals", "positive"],
        "cpa": ["CPA", "ratio", "amount", "goals", "negative"],
        "cr_users_created_ad": [
            "Конверсия в создавших кампанию",
            "share",
            "is_create_ad",
            "is_in_exp",
            "positive",
        ],
        "cr_users_with_spents": [
            "Конверсия в начавших тратить",
            "share",
            "is_with_spents",
            "is_in_exp",
            "positive",
        ],
        "camp_days_total": [
            "Суммарные дни открутки",
            "average",
            "camp_days_total",
            "camp_days_total",
            "positive",
        ],
        "amount_per_day": ["Amount per day", "average", "amount_per_day", "is_in_exp", "positive"],
    },
    orient="index",
    columns=["desc", "type", "num", "den", "direction"],
)

In [10]:
# Выполняем расчет теста
df_result_ab = run_ab_test(
    df=df_ab,
    metrics_df=METRICS_DF,
    exp_start_date=exp_start_date,
    exp_end_date=exp_end_date,
    group_base="A",
    group_exp="B",
    alpha=0.05,
    power=0.80,
)

In [11]:
# Опционально вывводим резы
display(
    style_table_ab(
        df_result_ab,
        comment=experiment_desc,
        exp_start_date=exp_start_date,
        exp_end_date=exp_end_date,
        experiment_id=142288,
    )
)

,metric_name,metric_type,value_base,value_exp,abs_delta,rel_delta,p_value,result,mde,number_samples,number_samples_base,number_samples_exp,avg_value_base,avg_value_exp,avg_abs_delta,avg_rel_delta,days_more_if_same_delta,days_more_base,days_more_exp,n_required_per_group,power_now,direction
0,Показы,additive,"10,955,130.0000","10,839,713.0000","-115,417.0000",-1.05%,0.3854,neutral,17.7378,66028,32914,33114,332.8410,327.3453,-5.4957,-1.65,76,76,76,343912,0.1398,positive
1,Клики,additive,"544,392.0000","543,080.0000","-1,312.0000",-0.24%,0.6979,neutral,1.0071,66028,32914,33114,16.5398,16.4003,-0.1395,-0.84,411,411,408,1720102,0.0674,positive
2,Списания,additive,"1,614,609.0803","8,153,601.1405","6,538,992.0602",404.99%,0.0000,positive,9.9032,66028,32914,33114,49.0554,246.2282,197.1728,401.94,0,0,0,84,1.0000,positive
3,Живые пополнения,average,96.2487,92.9865,-3.2622,-3.39%,0.1969,neutral,7.0810,5386,2650,2736,96.2487,92.9865,-3.2622,-3.39,31,31,30,12686,0.2522,positive
4,CPM,ratio,147.3838,752.1971,604.8133,410.37%,0.0000,negative,30.7371,66028,32914,33114,-0.0000,594.8269,594.8269,nan,0,0,0,86,1.0000,negative
5,CTR,ratio,0.0497,0.0501,0.0004,0.82%,0.7796,neutral,0.0040,66028,32914,33114,-0.0000,0.0004,0.0004,nan,770,770,766,3200206,0.0593,positive
6,CPC,ratio,2.9659,15.0136,12.0477,406.21%,0.0000,negative,0.6246,66028,32914,33114,0.0000,11.9461,11.9461,nan,0,0,0,89,1.0000,negative
7,Цели,additive,"47,203.0000","47,522.0000",319.0000,0.68%,0.9805,neutral,0.1111,66028,32914,33114,1.4341,1.4351,0.0010,0.07,104961,104961,104327,431866893,0.0501,positive
8,CPA,ratio,34.2056,171.5753,137.3696,401.60%,0.0000,negative,7.4053,66028,32914,33114,0.0000,137.4627,137.4627,nan,0,0,0,96,1.0000,negative
9,Конверсия в создавших кампанию,share,0.8946,0.8950,0.0004,0.04%,0.8687,neutral,0.0067,66028,32914,33114,nan,nan,nan,nan,2296,2296,2282,9478275,0.0531,positive


### 1.2 Подготовка отчета/сообщения

In [12]:
# Создаем отчет в Excel
excel_path = export_ab_results_to_excel(
    df_result_ab=df_result_ab,
    output_path="report_ab_exp_id_4242.xlsx",
    experiment_desc=experiment_desc,
    exp_id=experiment_id,
    date_from=exp_start_date,
    date_to=exp_end_date,
)

In [13]:
# Для редашика делаем ссылку на дашбрд, можно просто указать url любой
url_dash = build_dashboard_url_ab(
    base_url=base_url_ab,
    date_from=exp_start_date,
    date_to=exp_end_date,
    exp_id=experiment_id,
    extra_params={
        "p_is_business": "all",
        "p_is_novoreg": "all",
        "p_open_from": "all",
        "p_package_name": '["total"]',
        "p_platform": "all",
    },
)

In [ ]:
msg = build_ab_chat_message(
    df_result=df_result_ab,
    experiment_desc=experiment_desc,
    exp_id=experiment_id,
    date_from=exp_start_date,
    date_to=exp_end_date,
    dashboard_url=url_dash,
    max_metrics=10,
    show_days_more=True,
)

msg = msg + f"\n📄 Подробнее отчет: См. файл report_ab_exp_id_4242.xlsx"
print(msg)

### 1.3 LLM подключаем

In [15]:
# Дополняем анализ LLM если нужно
USER_PROMPT = msg

llm_text = build_llm_review(
    config=llm_config,
    system_prompt=SYSTEM_PROMPT,
    user_prompt=USER_PROMPT,
)

msg = msg + "\n\n" + llm_text

In [16]:
print(llm_text)

Вердикт: Нужен аналитик сейчас  
Оценка:  
- Данные о резком росте CPM, CPC, CPA и суммарных расходах указывают на потенциальную проблему с эффективностью рекламы, при этом показатели живых пополнений и других метрик не подтверждают стабильность роста.  
- Уровень риска: высокий


In [ ]:
print(msg)

### 1.4 Отправка сообщения

In [ ]:
# Отправка сообщение в чат
send_chat_message(
    config=chat_config,
    message=msg,
    chat_id=chat_id,
    file_path=excel_path,
)

## 2. ABC example

### 2.1 Расчет

In [19]:
# Передаем метрики, которые будем считать. Делаем настройки. Задаем тип метрики. Все как ниже
# Здесь можно менять:
# - отображаемое название метрики
# - тип расчёта
# - numerator / denominator
# - направление улучшения

METRICS_DF = pd.DataFrame.from_dict(
    {
        "shows": ["Показы", "additive", "shows", "shows", "positive"],
        "clicks": ["Клики", "additive", "clicks", "clicks", "positive"],
        "amount": ["Списания", "additive", "amount", "amount", "positive"],
        "a_amount_payment": [
            "Живые пополнения",
            "average",
            "a_amount_payment",
            "a_amount_payment",
            "positive",
        ],
        "cpm": ["CPM", "ratio", "amount_1000", "shows", "negative"],
        "ctr": ["CTR", "ratio", "clicks", "shows", "positive"],
        "cpc": ["CPC", "ratio", "amount", "clicks", "negative"],
        "goals": ["Цели", "additive", "goals", "goals", "positive"],
        "cpa": ["CPA", "ratio", "amount", "goals", "negative"],
        "cr_users_created_ad": [
            "Конверсия в создавших кампанию",
            "share",
            "is_create_ad",
            "is_in_exp",
            "positive",
        ],
        "cr_users_with_spents": [
            "Конверсия в начавших тратить",
            "share",
            "is_with_spents",
            "is_in_exp",
            "positive",
        ],
        "cr_users_created_to_spents": [
            "Конверсия из создавших в начавших тратить",
            "share",
            "is_with_spents",
            "is_create_ad",
            "positive",
        ],
        "camp_days_total": [
            "Суммарные дни открутки",
            "average",
            "camp_days_total",
            "camp_days_total",
            "positive",
        ],
        "camp_days_avg": [
            "Средние дни открутки",
            "average",
            "camp_days_avg",
            "is_in_exp",
            "positive",
        ],
        "camp_days_max": [
            "Campaign life days",
            "average",
            "camp_days_max",
            "is_in_exp",
            "positive",
        ],
        "amount_per_day": ["Amount per day", "average", "amount_per_day", "is_in_exp", "positive"],
        "goals_per_day": ["Goals per day", "average", "goals_per_day", "is_in_exp", "positive"],
    },
    orient="index",
    columns=["desc", "type", "num", "den", "direction"],
)

In [20]:
# Расчёт ABC-теста
df_result_abc = run_abc_test(
    df=df,
    metrics_df=METRICS_DF,
    exp_start_date=exp_start_date,
    exp_end_date=exp_end_date,
    include_bc=True,
    alpha=0.05,
    power=0.80,
    pvalue_adjust_method="holm",
)

In [21]:
# Опционально, вывод всего что вышло
style_table_abc(
    df_result_abc,
    caption=f"{experiment_desc}: {exp_start_date} — {exp_end_date}",
)

,metric_name,metric_type,pair,group_base,group_exp,value_base,value_exp,abs_delta,rel_delta,p_value,p_value_adj,result,result_adj,mde,number_samples,number_samples_base,number_samples_exp,avg_value_base,avg_value_exp,avg_abs_delta,avg_rel_delta,days_more_if_same_delta,days_more_base,days_more_exp,n_required_per_group,power_now,direction
0,Показы,additive,A_vs_B,A,B,"10,955,130.0000","10,839,713.0000","-115,417.0000",-1.05%,0.3854,1.0000,neutral,neutral,17.7378,66028,32914,33114,332.8410,327.3453,-5.4957,-1.65,76,76,76,343912,0.1398,positive
1,Показы,additive,A_vs_C,A,C,"10,955,130.0000","11,176,351.0000","221,221.0000",2.02%,0.5419,1.0000,neutral,neutral,17.6994,66886,32914,33972,332.8410,328.9871,-3.8539,-1.16,164,164,159,705202,0.0936,positive
2,Показы,additive,B_vs_C,B,C,"10,839,713.0000","11,176,351.0000","336,638.0000",3.11%,0.7929,1.0000,neutral,neutral,17.5227,67086,33114,33972,327.3453,328.9871,1.6418,0.50,915,915,892,3820205,0.0579,positive
3,Клики,additive,A_vs_B,A,B,"544,392.0000","543,080.0000","-1,312.0000",-0.24%,0.6979,1.0000,neutral,neutral,1.0071,66028,32914,33114,16.5398,16.4003,-0.1395,-0.84,411,411,408,1720102,0.0674,positive
4,Клики,additive,A_vs_C,A,C,"544,392.0000","549,685.0000","5,293.0000",0.97%,0.3137,0.9412,neutral,neutral,0.9991,66886,32914,33972,16.5398,16.1805,-0.3593,-2.17,55,55,53,258530,0.1719,positive
5,Клики,additive,B_vs_C,B,C,"543,080.0000","549,685.0000","6,605.0000",1.22%,0.5365,1.0000,neutral,neutral,0.9962,67086,33114,33972,16.4003,16.1805,-0.2198,-1.34,159,159,155,689084,0.0948,positive
6,Списания,additive,A_vs_B,A,B,"1,614,609.0803","8,153,601.1405","6,538,992.0602",404.99%,0.0000,0.0000,positive,positive,9.9032,66028,32914,33114,49.0554,246.2282,197.1728,401.94,0,0,0,84,1.0000,positive
7,Списания,additive,A_vs_C,A,C,"1,614,609.0803","1,677,826.6307","63,217.5504",3.92%,0.7279,0.7279,neutral,neutral,2.6825,66886,32914,33972,49.0554,49.3885,0.3331,0.68,519,519,503,2167990,0.0640,positive
8,Списания,additive,B_vs_C,B,C,"8,153,601.1405","1,677,826.6307","-6,475,774.5098",-79.42%,0.0000,0.0000,negative,negative,9.8241,67086,33114,33972,246.2282,49.3885,-196.8397,-79.94,0,0,0,84,1.0000,positive
9,Живые пополнения,average,A_vs_B,A,B,96.2487,92.9865,-3.2622,-3.39%,0.1969,0.5907,neutral,neutral,7.0810,5386,2650,2736,96.2487,92.9865,-3.2622,-3.39,31,31,30,12686,0.2522,positive


### 2.2 Подготовка отчета/сообщения

In [ ]:
# Формируем URL на дашборд с параметрами эксперимента (Только для Редаша, можно просто передать URL)
dashboard_url = build_dashboard_url_abc(
    base_url=base_url_abс,
    params={
        "p_date_start": exp_start_date,
        "p_date_end": exp_end_date,
        "p_exp_id": experiment_id,
        "p_is_business": "all",
        "p_is_novoreg": "all",
        "p_open_from": "all",
        "p_package_name": ["total"],
        "p_platform": "all",
    },
)

In [23]:
# Экспорт результата в Excel. файл сохраняется в корне проекта
excel_path = export_abc_results_to_excel(
    df_result_abc=df_result_abc,
    output_path="report_abc_id_4242.xlsx",
    experiment_desc=experiment_desc,
    exp_id=experiment_id,
    date_from=exp_start_date,
    date_to=exp_end_date,
)

In [ ]:
# Формируем короткое сообщение для чата / менеджера
msg = build_abc_chat_message(
    df_result=df_result_abc,
    experiment_desc=experiment_desc,
    exp_id=experiment_id,
    date_from=exp_start_date,
    date_to=exp_end_date,
    dashboard_url=dashboard_url,
    alpha=0.05,
    p_adjust_method="Holm",
    use_adjusted=True,
)

msg = msg + f"\n📄 Подробнее отчет: См. файл report_abc_id_4242.xlsx"
print(msg)

### 2.3 LLM подключаем

In [25]:
# Дополняем анализ LLM если нужно
USER_PROMPT = msg

llm_text = build_llm_review(
    config=llm_config,
    system_prompt=SYSTEM_PROMPT,
    user_prompt=USER_PROMPT,
)
msg = msg + "\n\n" + llm_text

In [26]:
print(llm_text)



Вердикт: Нужен аналитик сейчас

Оценка:  
- Падение ключевых финансовых метрик (Суммарные списания, CPM, CPC, Сумма в день) в группе B на 79–80% по сравнению с C, с высокой статистической значимостью.  
- Уровень риска: высокий


In [ ]:
print(msg)

### 2.4 Отправка сообщения

In [ ]:
# Отправка сообщение в чат
send_chat_message(
    config=chat_config,
    message=msg,
    chat_id=chat_id,
    file_path=excel_path,
)

## 3. Duration planning

Раздел показывает, как оценить:
- какой MDE ожидается при разной длительности эксперимента
- сколько дней нужно, чтобы достичь target MDE
- как собрать итоговый план в Excel

### 3.1 Даннные для расчета

In [29]:
# датасет который надо будет выгрузить для расчета
# это просто пример функции. Пиши свой на SQL/YQL

import numpy as np


def make_synthetic_campaign_daily_data(
    n_rows: int = 4242, seed: int = 42, start_date: str = "2025-01-01", end_date: str = "2025-12-31"
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    dates = pd.date_range(start=start_date, end=end_date, freq="D")
    n_days = len(dates)
    rows_per_day = int(np.ceil(n_rows / n_days))
    data = {
        "date": np.repeat(dates.date, rows_per_day)[:n_rows],
        "user_id": rng.integers(12_000_000, 30_000_000, size=n_rows),
        "campaign_id": rng.integers(105_000_000, 127_000_000, size=n_rows),
        "package_id": rng.choice(
            [3194, 3122, 3785, 2841, 3650], size=n_rows, p=[0.65, 0.15, 0.10, 0.07, 0.03]
        ),
        "campaign_create_date": None,
        "camp_date_end": None,
        "amount": np.round(
            rng.exponential(scale=280, size=n_rows) * rng.binomial(1, 0.92, size=n_rows), 4
        ),
        "shows": rng.integers(50, 45_000, size=n_rows),
        "clicks": rng.integers(0, 2800, size=n_rows),
        "main_goals": rng.integers(0, 1200, size=n_rows),
        "campaign_day": rng.integers(1, 32, size=n_rows),
    }
    df = pd.DataFrame(data)
    df["campaign_create_date"] = pd.to_datetime(df["date"]) - pd.to_timedelta(
        df["campaign_day"] - 1, unit="D"
    )
    campaign_duration = rng.integers(3, 61, size=n_rows)
    df["camp_date_end"] = df["campaign_create_date"] + pd.to_timedelta(campaign_duration, unit="D")
    df["campaign_create_date"] = pd.to_datetime(df["campaign_create_date"])
    df["camp_date_end"] = pd.to_datetime(df["camp_date_end"])
    df["date"] = pd.to_datetime(df["date"])
    zero_mask = rng.binomial(1, 0.08, size=n_rows) == 1
    df.loc[zero_mask, ["amount", "shows", "clicks", "main_goals"]] = 0
    df["clicks"] = np.minimum(df["clicks"], df["shows"])
    df["main_goals"] = np.minimum(df["main_goals"], df["clicks"] * 2)
    df = df.sort_values(by=["date", "campaign_id"]).reset_index(drop=True)
    df["amount"] = df["amount"].round(4)
    return df


df_raw = make_synthetic_campaign_daily_data(n_rows=100_000, seed=42)

In [ ]:
# Это то что считается по стандрату
metrics_duration_df = default_duration_metrics_config()
display(metrics_duration_df)

,label,source_col,transform
shows,"Показы MDE, %",total_shows,none
clicks,"Клики MDE, %",total_clicks,none
amount,"Списания MDE, %",total_amount,none
goals,"Цели MDE, %",total_goals,none
cpm,"CPM MDE, %",cpm,none
ctr,"CTR MDE, %",ctr,none
cpa,"CPA MDE, %",cpa,none
life_days,"Life days MDE, %",life_days,none
amount_per_day,"Amount per day MDE, %",amount_per_day,none
goals_per_day,"Goals per day MDE, %",goals_per_day,none


### 3.2 Строим агрегат на нужном уровне

In [31]:
# =========================
# 1) Строим агрегат на нужном уровне
# Например, на уровне user_id
# =========================
agg_df = build_experiment_level(
    df_raw,
    unit_id_col="user_id",
    date_col="date",
    amount_col="amount",
    shows_col="shows",
    clicks_col="clicks",
    goals_col="main_goals",
    entity_id_col="campaign_id",  # сколько уникальных campaign_id внутри user_id
)

display(agg_df.head())

,user_id,start_date,end_date,active_days,total_amount,total_shows,total_clicks,total_goals,entities_cnt,life_days,cpm,ctr,cpa,amount_per_day,goals_per_day,is_with_spend,is_with_goals
0,12000015,2025-01-19,2025-01-19,1,141.9022,8715,2210,852,1,1,16.2825,0.2536,0.1666,141.9022,852.0000,1,1
1,12000054,2025-07-01,2025-07-01,1,128.1175,33980,2531,1109,1,1,3.7704,0.0745,0.1155,128.1175,"1,109.0000",1,1
2,12000237,2025-03-16,2025-03-16,1,218.3621,18269,177,204,1,1,11.9526,0.0097,1.0704,218.3621,204.0000,1,1
3,12000331,2025-09-03,2025-09-03,1,123.2627,27470,1889,46,1,1,4.4872,0.0688,2.6796,123.2627,46.0000,1,1
4,12000346,2025-05-19,2025-05-19,1,3.0594,37241,1510,381,1,1,0.0822,0.0405,0.0080,3.0594,381.0000,1,1


### 3.3 MDE по длительности эксперимента — AB

In [32]:
# =========================
# 2) MDE по длительности эксперимента — AB
# =========================
mde_ab = mde_table_for_experiment_duration(
    df_raw=df_raw,
    unit_id_col="user_id",
    rollout_pct=0.5,
    exp_days=[7, 14, 21, 28, 35],
    experiment_type="ab",
    metrics_config=metrics_duration_df,
    alpha=0.05,
    power=0.80,
    date_col="date",
    amount_col="amount",
    shows_col="shows",
    clicks_col="clicks",
    goals_col="main_goals",
    entity_id_col="campaign_id",
)

display(mde_ab)

,% Выкатки,Число дней экспа,Тип эксперимента,Число групп,"Показы MDE, %","Клики MDE, %","Списания MDE, %","Цели MDE, %","CPM MDE, %","CTR MDE, %","CPA MDE, %","Life days MDE, %","Amount per day MDE, %","Goals per day MDE, %","Active days MDE, %","Entities count MDE, %",_units_per_day,_eligible_per_day,_smallest_group_share,_n_per_group,_alpha,_power,_group_shares
0,50.0000,7,AB,2,12.1083,12.3343,21.1220,12.7982,103.6500,28.2939,146.0160,104.6017,21.1242,12.7682,0.9580,0.9580,273.2082,136.6041,0.5000,478,0.0500,0.8000,"0.5000,0.5000"
1,50.0000,14,AB,2,8.5618,8.7216,14.9355,9.0497,73.2916,20.0068,103.2489,73.9646,14.9371,9.0285,0.6774,0.6774,273.2082,136.6041,0.5000,956,0.0500,0.8000,"0.5000,0.5000"
2,50.0000,21,AB,2,6.9907,7.1212,12.1948,7.3891,59.8424,16.3355,84.3024,60.3918,12.1961,7.3718,0.5531,0.5531,273.2082,136.6041,0.5000,1434,0.0500,0.8000,"0.5000,0.5000"
3,50.0000,28,AB,2,6.0541,6.1671,10.5610,6.3991,51.8250,14.1469,73.0080,52.3009,10.5621,6.3841,0.4790,0.4790,273.2082,136.6041,0.5000,1912,0.0500,0.8000,"0.5000,0.5000"
4,50.0000,35,AB,2,5.4150,5.5160,9.4461,5.7235,46.3537,12.6534,65.3003,46.7793,9.4470,5.7101,0.4284,0.4284,273.2082,136.6041,0.5000,2390,0.0500,0.8000,"0.5000,0.5000"


### 3.3 MDE по длительности эксперимента — ABC

In [33]:
# =========================
# 3) MDE по длительности эксперимента — ABC
# =========================
mde_abc = mde_table_for_experiment_duration(
    df_raw=df_raw,
    unit_id_col="user_id",
    rollout_pct=0.5,
    exp_days=[7, 14, 21, 28, 35],
    experiment_type="abc",
    metrics_config=metrics_duration_df,
    alpha=0.05,
    power=0.80,
    date_col="date",
    amount_col="amount",
    shows_col="shows",
    clicks_col="clicks",
    goals_col="main_goals",
    entity_id_col="campaign_id",
)

display(mde_abc)

,% Выкатки,Число дней экспа,Тип эксперимента,Число групп,"Показы MDE, %","Клики MDE, %","Списания MDE, %","Цели MDE, %","CPM MDE, %","CTR MDE, %","CPA MDE, %","Life days MDE, %","Amount per day MDE, %","Goals per day MDE, %","Active days MDE, %","Entities count MDE, %",_units_per_day,_eligible_per_day,_smallest_group_share,_n_per_group,_alpha,_power,_group_shares
0,50.0000,7,ABC,3,14.8451,15.1221,25.8962,15.6910,127.0778,34.6891,179.0197,128.2446,25.8989,15.6542,1.1745,1.1745,273.2082,136.6041,0.3333,318,0.0500,0.8000,"0.3333,0.3333,0.3333"
1,50.0000,14,ABC,3,10.4888,10.6846,18.2970,11.0865,89.7870,24.5096,126.4866,90.6114,18.2989,11.0605,0.8298,0.8298,273.2082,136.6041,0.3333,637,0.0500,0.8000,"0.3333,0.3333,0.3333"
2,50.0000,21,ABC,3,8.5618,8.7216,14.9355,9.0497,73.2916,20.0068,103.2489,73.9646,14.9371,9.0285,0.6774,0.6774,273.2082,136.6041,0.3333,956,0.0500,0.8000,"0.3333,0.3333,0.3333"
3,50.0000,28,ABC,3,7.4167,7.5551,12.9379,7.8393,63.4890,17.3309,89.4395,64.0720,12.9393,7.8210,0.5868,0.5868,273.2082,136.6041,0.3333,1274,0.0500,0.8000,"0.3333,0.3333,0.3333"
4,50.0000,35,ABC,3,6.6327,6.7565,11.5702,7.0106,56.7774,15.4988,79.9846,57.2987,11.5714,6.9942,0.5248,0.5248,273.2082,136.6041,0.3333,1593,0.0500,0.8000,"0.3333,0.3333,0.3333"


### 3.4 Неравные доли групп

In [34]:
# =========================
# 4) Неравные доли групп
# =========================
mde_abc_custom = mde_table_for_experiment_duration(
    df_raw=df_raw,
    unit_id_col="user_id",
    rollout_pct=0.5,
    exp_days=[7, 14, 21, 28, 35],
    experiment_type="abc",
    group_shares=[0.50, 0.25, 0.25],
    metrics_config=metrics_duration_df,
    date_col="date",
    amount_col="amount",
    shows_col="shows",
    clicks_col="clicks",
    goals_col="main_goals",
    entity_id_col="campaign_id",
)

display(mde_abc_custom)

,% Выкатки,Число дней экспа,Тип эксперимента,Число групп,"Показы MDE, %","Клики MDE, %","Списания MDE, %","Цели MDE, %","CPM MDE, %","CTR MDE, %","CPA MDE, %","Life days MDE, %","Amount per day MDE, %","Goals per day MDE, %","Active days MDE, %","Entities count MDE, %",_units_per_day,_eligible_per_day,_smallest_group_share,_n_per_group,_alpha,_power,_group_shares
0,50.0000,7,ABC,3,17.1237,17.4433,29.8711,18.0994,146.5832,40.0136,206.4978,147.9292,29.8742,18.0570,1.3548,1.3548,273.2082,136.6041,0.2500,239,0.0500,0.8000,"0.5000,0.2500,0.2500"
1,50.0000,14,ABC,3,12.1083,12.3343,21.1220,12.7982,103.6500,28.2939,146.0160,104.6017,21.1242,12.7682,0.9580,0.9580,273.2082,136.6041,0.2500,478,0.0500,0.8000,"0.5000,0.2500,0.2500"
2,50.0000,21,ABC,3,9.8864,10.0709,17.2461,10.4497,84.6299,23.1019,119.2215,85.4069,17.2479,10.4252,0.7822,0.7822,273.2082,136.6041,0.2500,717,0.0500,0.8000,"0.5000,0.2500,0.2500"
3,50.0000,28,ABC,3,8.5618,8.7216,14.9355,9.0497,73.2916,20.0068,103.2489,73.9646,14.9371,9.0285,0.6774,0.6774,273.2082,136.6041,0.2500,956,0.0500,0.8000,"0.5000,0.2500,0.2500"
4,50.0000,35,ABC,3,7.6579,7.8009,13.3588,8.0943,65.5540,17.8946,92.3486,66.1559,13.3601,8.0753,0.6059,0.6059,273.2082,136.6041,0.2500,1195,0.0500,0.8000,"0.5000,0.2500,0.2500"


### 3.5 Сколько дней нужно до target MDE - AB

In [35]:
# =========================
# 5) Сколько дней нужно до target MDE - AB
# =========================
days_to_target_ab = required_days_for_target_mde_table(
    df_raw=df_raw,
    unit_id_col="user_id",
    rollout_pct=0.5,
    target_mde_pct=[5, 7, 10],
    experiment_type="ab",
    metrics_config=metrics_duration_df,
    alpha=0.05,
    power=0.80,
    date_col="date",
    amount_col="amount",
    shows_col="shows",
    clicks_col="clicks",
    goals_col="main_goals",
    entity_id_col="campaign_id",
)

display(days_to_target_ab)

,% Выкатки,"Target MDE, %",Тип эксперимента,Число групп,Показы — дней до target MDE,Показы — n_per_group,Клики — дней до target MDE,Клики — n_per_group,Списания — дней до target MDE,Списания — n_per_group,Цели — дней до target MDE,Цели — n_per_group,CPM — дней до target MDE,CPM — n_per_group,CTR — дней до target MDE,CTR — n_per_group,CPA — дней до target MDE,CPA — n_per_group,Life days — дней до target MDE,Life days — n_per_group,Amount per day — дней до target MDE,Amount per day — n_per_group,Goals per day — дней до target MDE,Goals per day — n_per_group,Active days — дней до target MDE,Active days — n_per_group,Entities count — дней до target MDE,Entities count — n_per_group,_units_per_day,_eligible_per_day,_smallest_group_share,_alpha,_power,_group_shares
0,50.0000,5.0000,AB,2,42,"2,804.0000",43,"2,909.0000",125,"8,531.0000",46,"3,132.0000",3008,"205,413.0000",225,"15,307.0000",5969,"407,652.0000",3063,"209,202.0000",125,"8,532.0000",46,"3,118.0000",1,18.0000,1,18.0000,273.2082,136.6041,0.5000,0.0500,0.8000,"0.5000,0.5000"
1,50.0000,7.0000,AB,2,21,"1,431.0000",22,"1,485.0000",64,"4,353.0000",24,"1,598.0000",1535,"104,803.0000",115,"7,810.0000",3046,"207,986.0000",1563,"106,736.0000",64,"4,354.0000",24,"1,591.0000",1,9.0000,1,9.0000,273.2082,136.6041,0.5000,0.0500,0.8000,"0.5000,0.5000"
2,50.0000,10.0000,AB,2,11,701.0000,11,728.0000,32,"2,133.0000",12,783.0000,752,"51,354.0000",57,"3,827.0000",1493,"101,913.0000",766,"52,301.0000",32,"2,133.0000",12,780.0000,1,5.0000,1,5.0000,273.2082,136.6041,0.5000,0.0500,0.8000,"0.5000,0.5000"


### 3.6 Сколько дней нужно до target MDE - ABC

In [36]:
# =========================
# 6) Сколько дней нужно до target MDE - ABC
# =========================
days_to_target_abc = required_days_for_target_mde_table(
    df_raw=df_raw,
    unit_id_col="user_id",
    rollout_pct=0.5,
    target_mde_pct=[5, 7, 10],
    experiment_type="abc",
    metrics_config=metrics_duration_df,
    date_col="date",
    amount_col="amount",
    shows_col="shows",
    clicks_col="clicks",
    goals_col="main_goals",
    entity_id_col="campaign_id",
)

display(days_to_target_abc)

,% Выкатки,"Target MDE, %",Тип эксперимента,Число групп,Показы — дней до target MDE,Показы — n_per_group,Клики — дней до target MDE,Клики — n_per_group,Списания — дней до target MDE,Списания — n_per_group,Цели — дней до target MDE,Цели — n_per_group,CPM — дней до target MDE,CPM — n_per_group,CTR — дней до target MDE,CTR — n_per_group,CPA — дней до target MDE,CPA — n_per_group,Life days — дней до target MDE,Life days — n_per_group,Amount per day — дней до target MDE,Amount per day — n_per_group,Goals per day — дней до target MDE,Goals per day — n_per_group,Active days — дней до target MDE,Active days — n_per_group,Entities count — дней до target MDE,Entities count — n_per_group,_units_per_day,_eligible_per_day,_smallest_group_share,_alpha,_power,_group_shares
0,50.0000,5.0000,ABC,3,62,"2,804.0000",64,"2,909.0000",188,"8,531.0000",69,"3,132.0000",4512,"205,413.0000",337,"15,307.0000",8953,"407,652.0000",4595,"209,202.0000",188,"8,532.0000",69,"3,118.0000",1,18.0000,1,18.0000,273.2082,136.6041,0.3333,0.0500,0.8000,"0.3333,0.3333,0.3333"
1,50.0000,7.0000,ABC,3,32,"1,431.0000",33,"1,485.0000",96,"4,353.0000",36,"1,598.0000",2302,"104,803.0000",172,"7,810.0000",4568,"207,986.0000",2345,"106,736.0000",96,"4,354.0000",35,"1,591.0000",1,9.0000,1,9.0000,273.2082,136.6041,0.3333,0.0500,0.8000,"0.3333,0.3333,0.3333"
2,50.0000,10.0000,ABC,3,16,701.0000,16,728.0000,47,"2,133.0000",18,783.0000,1128,"51,354.0000",85,"3,827.0000",2239,"101,913.0000",1149,"52,301.0000",47,"2,133.0000",18,780.0000,1,5.0000,1,5.0000,273.2082,136.6041,0.3333,0.0500,0.8000,"0.3333,0.3333,0.3333"


### 3.7 Общий summary

In [37]:
# =========================
# 7) Общий summary
# =========================
plan = duration_plan_summary(
    df_raw=df_raw,
    unit_id_col="user_id",
    rollout_pct=0.5,
    exp_days=[7, 14, 21, 28, 35],
    target_mde_pct=[5, 7, 10],
    experiment_type="ab",
    metrics_config=metrics_duration_df,
    alpha=0.05,
    power=0.80,
    date_col="date",
    amount_col="amount",
    shows_col="shows",
    clicks_col="clicks",
    goals_col="main_goals",
    entity_id_col="campaign_id",
)

display(plan["aggregated_df"].head())
display(plan["mde_by_days"])
display(plan["days_for_target_mde"])

,user_id,start_date,end_date,active_days,total_amount,total_shows,total_clicks,total_goals,entities_cnt,life_days,cpm,ctr,cpa,amount_per_day,goals_per_day,is_with_spend,is_with_goals
0,12000015,2025-01-19,2025-01-19,1,141.9022,8715,2210,852,1,1,16.2825,0.2536,0.1666,141.9022,852.0000,1,1
1,12000054,2025-07-01,2025-07-01,1,128.1175,33980,2531,1109,1,1,3.7704,0.0745,0.1155,128.1175,"1,109.0000",1,1
2,12000237,2025-03-16,2025-03-16,1,218.3621,18269,177,204,1,1,11.9526,0.0097,1.0704,218.3621,204.0000,1,1
3,12000331,2025-09-03,2025-09-03,1,123.2627,27470,1889,46,1,1,4.4872,0.0688,2.6796,123.2627,46.0000,1,1
4,12000346,2025-05-19,2025-05-19,1,3.0594,37241,1510,381,1,1,0.0822,0.0405,0.0080,3.0594,381.0000,1,1


,% Выкатки,Число дней экспа,Тип эксперимента,Число групп,"Показы MDE, %","Клики MDE, %","Списания MDE, %","Цели MDE, %","CPM MDE, %","CTR MDE, %","CPA MDE, %","Life days MDE, %","Amount per day MDE, %","Goals per day MDE, %","Active days MDE, %","Entities count MDE, %",_units_per_day,_eligible_per_day,_smallest_group_share,_n_per_group,_alpha,_power,_group_shares
0,50.0000,7,AB,2,12.1083,12.3343,21.1220,12.7982,103.6500,28.2939,146.0160,104.6017,21.1242,12.7682,0.9580,0.9580,273.2082,136.6041,0.5000,478,0.0500,0.8000,"0.5000,0.5000"
1,50.0000,14,AB,2,8.5618,8.7216,14.9355,9.0497,73.2916,20.0068,103.2489,73.9646,14.9371,9.0285,0.6774,0.6774,273.2082,136.6041,0.5000,956,0.0500,0.8000,"0.5000,0.5000"
2,50.0000,21,AB,2,6.9907,7.1212,12.1948,7.3891,59.8424,16.3355,84.3024,60.3918,12.1961,7.3718,0.5531,0.5531,273.2082,136.6041,0.5000,1434,0.0500,0.8000,"0.5000,0.5000"
3,50.0000,28,AB,2,6.0541,6.1671,10.5610,6.3991,51.8250,14.1469,73.0080,52.3009,10.5621,6.3841,0.4790,0.4790,273.2082,136.6041,0.5000,1912,0.0500,0.8000,"0.5000,0.5000"
4,50.0000,35,AB,2,5.4150,5.5160,9.4461,5.7235,46.3537,12.6534,65.3003,46.7793,9.4470,5.7101,0.4284,0.4284,273.2082,136.6041,0.5000,2390,0.0500,0.8000,"0.5000,0.5000"


,% Выкатки,"Target MDE, %",Тип эксперимента,Число групп,Показы — дней до target MDE,Показы — n_per_group,Клики — дней до target MDE,Клики — n_per_group,Списания — дней до target MDE,Списания — n_per_group,Цели — дней до target MDE,Цели — n_per_group,CPM — дней до target MDE,CPM — n_per_group,CTR — дней до target MDE,CTR — n_per_group,CPA — дней до target MDE,CPA — n_per_group,Life days — дней до target MDE,Life days — n_per_group,Amount per day — дней до target MDE,Amount per day — n_per_group,Goals per day — дней до target MDE,Goals per day — n_per_group,Active days — дней до target MDE,Active days — n_per_group,Entities count — дней до target MDE,Entities count — n_per_group,_units_per_day,_eligible_per_day,_smallest_group_share,_alpha,_power,_group_shares
0,50.0000,5.0000,AB,2,42,"2,804.0000",43,"2,909.0000",125,"8,531.0000",46,"3,132.0000",3008,"205,413.0000",225,"15,307.0000",5969,"407,652.0000",3063,"209,202.0000",125,"8,532.0000",46,"3,118.0000",1,18.0000,1,18.0000,273.2082,136.6041,0.5000,0.0500,0.8000,"0.5000,0.5000"
1,50.0000,7.0000,AB,2,21,"1,431.0000",22,"1,485.0000",64,"4,353.0000",24,"1,598.0000",1535,"104,803.0000",115,"7,810.0000",3046,"207,986.0000",1563,"106,736.0000",64,"4,354.0000",24,"1,591.0000",1,9.0000,1,9.0000,273.2082,136.6041,0.5000,0.0500,0.8000,"0.5000,0.5000"
2,50.0000,10.0000,AB,2,11,701.0000,11,728.0000,32,"2,133.0000",12,783.0000,752,"51,354.0000",57,"3,827.0000",1493,"101,913.0000",766,"52,301.0000",32,"2,133.0000",12,780.0000,1,5.0000,1,5.0000,273.2082,136.6041,0.5000,0.0500,0.8000,"0.5000,0.5000"


### 3.8 Пример: расширяем дефолтный конфиг своими метриками

In [38]:
# =========================
# 8) Пример: расширяем дефолтный конфиг своими метриками
# =========================
metrics_duration_custom = default_duration_metrics_config().copy()

metrics_duration_custom.loc["amount_log"] = {
    "label": "Списания log1p MDE, %",
    "source_col": "total_amount",
    "transform": "log1p",
}

metrics_duration_custom.loc["custom_life"] = {
    "label": "Life days MDE, %",
    "source_col": "life_days",
    "transform": "none",
}

mde_custom = mde_table_for_experiment_duration(
    df_raw=df_raw,
    unit_id_col="user_id",
    rollout_pct=0.5,
    exp_days=[7, 14, 21, 28],
    experiment_type="abc",
    metrics_config=metrics_duration_custom,
    date_col="date",
    amount_col="amount",
    shows_col="shows",
    clicks_col="clicks",
    goals_col="main_goals",
    entity_id_col="campaign_id",
)

display(mde_custom)

,% Выкатки,Число дней экспа,Тип эксперимента,Число групп,"Показы MDE, %","Клики MDE, %","Списания MDE, %","Цели MDE, %","CPM MDE, %","CTR MDE, %","CPA MDE, %","Life days MDE, %","Amount per day MDE, %","Goals per day MDE, %","Active days MDE, %","Entities count MDE, %","Списания log1p MDE, %",_units_per_day,_eligible_per_day,_smallest_group_share,_n_per_group,_alpha,_power,_group_shares
0,50.0000,7,ABC,3,14.8451,15.1221,25.8962,15.6910,127.0778,34.6891,179.0197,128.2446,25.8989,15.6542,1.1745,1.1745,11.0364,273.2082,136.6041,0.3333,318,0.0500,0.8000,"0.3333,0.3333,0.3333"
1,50.0000,14,ABC,3,10.4888,10.6846,18.2970,11.0865,89.7870,24.5096,126.4866,90.6114,18.2989,11.0605,0.8298,0.8298,7.7978,273.2082,136.6041,0.3333,637,0.0500,0.8000,"0.3333,0.3333,0.3333"
2,50.0000,21,ABC,3,8.5618,8.7216,14.9355,9.0497,73.2916,20.0068,103.2489,73.9646,14.9371,9.0285,0.6774,0.6774,6.3652,273.2082,136.6041,0.3333,956,0.0500,0.8000,"0.3333,0.3333,0.3333"
3,50.0000,28,ABC,3,7.4167,7.5551,12.9379,7.8393,63.4890,17.3309,89.4395,64.0720,12.9393,7.8210,0.5868,0.5868,5.5138,273.2082,136.6041,0.3333,1274,0.0500,0.8000,"0.3333,0.3333,0.3333"


### 3.9 Экспорт результата в Excel (готовый ответ для задачи)

In [39]:
# =========================
# 9) Экспорт результата в Excel
# =========================
metrics_duration_df = default_duration_metrics_config()

mde_ab = mde_table_for_experiment_duration(
    df_raw=df_raw,
    unit_id_col="user_id",
    rollout_pct=0.5,
    exp_days=[7, 14, 21, 28, 35],
    experiment_type="ab",
    metrics_config=metrics_duration_df,
    alpha=0.05,
    power=0.80,
    date_col="date",
    amount_col="amount",
    shows_col="shows",
    clicks_col="clicks",
    goals_col="main_goals",
    entity_id_col="campaign_id",
)

days_to_target_ab = required_days_for_target_mde_table(
    df_raw=df_raw,
    unit_id_col="user_id",
    rollout_pct=0.5,
    target_mde_pct=[5, 7, 10],
    experiment_type="ab",
    metrics_config=metrics_duration_df,
    alpha=0.05,
    power=0.80,
    date_col="date",
    amount_col="amount",
    shows_col="shows",
    clicks_col="clicks",
    goals_col="main_goals",
    entity_id_col="campaign_id",
)

excel_path = export_duration_results_to_excel(
    output_path="duration_plan_ab.xlsx",
    mde_by_days_df=mde_ab,
    days_for_target_mde_df=days_to_target_ab,
    experiment_name="Новые рекомендации бюджета в Слое",
    experiment_type="ab",
    rollout_pct=50,
    recommended_days=21,
    comment="Оценка выполнена на исторических данных на уровне user_id.",
)

print(excel_path)

duration_plan_ab.xlsx


In [40]:
# =========================
# 10) Вариант со своим текстом в Ексельке
# =========================
custom_summary = """
Планирование длительности эксперимента: Новые рекомендации бюджета в Слое
тип: AB / выкатка: 50,0% / рекомендуемая длительность: 21 дн.

Что видно по расчёту:
- При длительности ~21 дн. ожидаемый MDE по списаниям около 10%, по CPA около 7,8%.
- Для более мелкого эффекта тест, вероятно, придётся катить дольше 21 дня.

Интерпретация:
- Текущая чувствительность достаточна для medium-size эффекта.
- Для слабых эффектов лучше либо продлевать тест, либо увеличивать выкладку.
"""

excel_path = export_duration_results_to_excel(
    output_path="duration_plan_ab_custom.xlsx",
    mde_by_days_df=mde_ab,
    days_for_target_mde_df=days_to_target_ab,
    experiment_name="Новые рекомендации бюджета в Слое",
    summary_text=custom_summary,
)

print(excel_path)

duration_plan_ab_custom.xlsx
